# Microsoft Teams NPS Data Cleaning

This notebook creates a clean analytical dataset from `Dataset.csv`. The original file is preserved.

Cleaning decisions:
- Remove fully empty columns, sparse Excel export-artifact columns, and exact duplicate columns.
- Keep CPU architecture excluded because it is almost entirely missing and cannot be reliably analyzed.
- Parse date fields without filling missing values.
- Preserve categorical and numeric missing values for EDA; no automatic `Unknown` filling, median imputation, or missingness flags.
- Validate ratings, response labels, duplicate rows, and duplicate feedback IDs.
- Save the result as `Dataset_clean.csv`.

In [7]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

INPUT_FILE = 'Dataset.csv'
OUTPUT_FILE = 'Dataset_clean.csv'
raw = pd.read_csv(INPUT_FILE)
clean = raw.copy()
print(f'Original shape: {clean.shape}')

Original shape: (31469, 64)


## 1. Remove export artifacts and duplicate fields

In [8]:
empty_columns = clean.columns[clean.isna().all()].tolist()
sparse_export_columns = [
    column for column in clean.columns
    if column.startswith('Unnamed') and clean[column].notna().sum() < 100
]
artifact_columns = sorted(set(empty_columns + sparse_export_columns))
clean = clean.drop(columns=artifact_columns, errors='ignore')

duplicate_column_pairs = []
for left_index, left in enumerate(clean.columns):
    for right in clean.columns[left_index + 1:]:
        if clean[left].equals(clean[right]):
            duplicate_column_pairs.append((left, right))

duplicate_columns_to_drop = []
for left, right in duplicate_column_pairs:
    if right.endswith('.1'):
        duplicate_columns_to_drop.append(right)

clean = clean.drop(columns=duplicate_columns_to_drop, errors='ignore')
print(f'Removed {len(artifact_columns)} export-artifact columns')
print(f'Removed duplicate columns: {duplicate_columns_to_drop}')
print(f'Shape after structural cleaning: {clean.shape}')

Removed 35 export-artifact columns
Removed duplicate columns: ['PipelineInfo_ClientCountry.1', 'Tenant_Country.1']
Shape after structural cleaning: (31469, 27)


## 2. Parse dates and standardize categorical fields

In [9]:
date_columns = ['Feedback_DateTime', 'FirstSeenTime_Date']
for column in date_columns:
    clean[column] = pd.to_datetime(clean[column], errors='coerce')

categorical_columns = [
    'ResponseType', 'Feedback_Month', 'NewUser_Period',
    'PipelineInfo_ClientCountry', 'Tenant_Country',
    'Tenant_MSSalesVerticalName', 'Tenant_CustomerSegmentGroup'
]
categorical_columns = [column for column in categorical_columns if column in clean.columns]
for column in categorical_columns:
    clean[column] = clean[column].astype('string').str.strip()

print('Date columns parsed:', date_columns)
print('Categorical and numeric missing values preserved for EDA')

Date columns parsed: ['Feedback_DateTime', 'FirstSeenTime_Date']
Categorical and numeric missing values preserved for EDA


## 3. Handle numeric missing values

In [10]:
# CPU architecture is 99.75% missing, so it is excluded rather than imputed.
unusable_columns = [
    column for column in ['R7_DeviceInfo_CpuArchitecture']
    if column in clean.columns
]
clean = clean.drop(columns=unusable_columns)

numeric_columns = clean.select_dtypes(include='number').columns.tolist()
numeric_missing_summary = clean[numeric_columns].isna().sum().sort_values(ascending=False)

print('Dropped near-empty fields:', unusable_columns)
print('Numeric missing values preserved:', int(numeric_missing_summary.sum()))
display(numeric_missing_summary[numeric_missing_summary > 0].to_frame('Missing values preserved for EDA'))

Dropped near-empty fields: ['R7_DeviceInfo_CpuArchitecture']
Numeric missing values preserved: 3402


,Missing values preserved for EDA
R7_DeviceInfo_CpuSpeed,1715
R7_DeviceInfo_CpuCores,1687


## 4. Validate the cleaned dataset

In [11]:
derived_response = pd.cut(
    clean['Feedback_Rating'],
    bins=[-1, 6, 8, 10],
    labels=['Detractor', 'Passive', 'Promoter']
).astype('string')

validation = pd.DataFrame({
    'Check': [
        'Duplicate rows',
        'Duplicate Feedback_ID values',
        'Invalid ratings',
        'Response label mismatches',
        'Remaining missing values',
    ],
    'Result': [
        clean.duplicated().sum(),
        clean['Feedback_ID'].duplicated().sum(),
        ((clean['Feedback_Rating'] < 0) | (clean['Feedback_Rating'] > 10)).sum(),
        (clean['ResponseType'].astype('string') != derived_response).sum(),
        int(clean.isna().sum().sum()),
    ]
})
display(validation)
assert validation.loc[0, 'Result'] == 0
assert validation.loc[1, 'Result'] == 0
assert validation.loc[2, 'Result'] == 0
assert validation.loc[3, 'Result'] == 0

,Check,Result
0,Duplicate rows,0
1,Duplicate Feedback_ID values,0
2,Invalid ratings,0
3,Response label mismatches,0
4,Remaining missing values,4108


## 5. Save the clean dataset

In [12]:
clean.to_csv(OUTPUT_FILE, index=False)
print(f'Clean dataset saved to: {OUTPUT_FILE}')
print(f'Final shape: {clean.shape}')
print(f'Remaining missing values: {int(clean.isna().sum().sum())}')

Clean dataset saved to: Dataset_clean.csv
Final shape: (31469, 26)
Remaining missing values: 4108
